# Royal Champion Walk — Clash of Clans RL attack agentA DQN that learns to run a **Royal Champion charge** on a Town Hall 15 base:where to deploy her, when and where to drop Invisibility Spells, and when topop Seeking Shield. Her walking and target selection are the game's hero AI —the agent only plays the human's part of the attack.The heavy lifting lives in `src/coc/`; this notebook just drives it.| module | what it holds ||---|---|| `config.py` | every game constant, sourced from the wiki, plus the curriculum || `base.py` | procedural TH15 base generation || `env.py` | the attack simulation || `model.py` | the dueling, fully-convolutional Q-network (~53k params) || `replay.py` | n-step replay buffer, uint8-compressed and checkpointable || `train.py` | Double-DQN training loop with curriculum and real resume || `evaluate.py` | evaluation against three baseline policies || `viz.py` | base rendering, battle replay GIF, learning curves |

In [ ]:
import sys, os, pathlibROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()sys.path.insert(0, str(ROOT / "src"))import numpy as np, torch, matplotlib.pyplot as pltfrom coc import config as Cfrom coc.env import RCWalkEnvfrom coc.model import RCQNet, count_parametersfrom coc.train import Trainer, pick_devicefrom coc.evaluate import compare, run_policy, policy_scripted_human, make_model_policyfrom coc.viz import render_base, animate_battle, plot_trainingprint("device:", pick_device("auto"))print("action space:", C.N_ACTIONS, "(wait, Seeking Shield, + 22x22 spell tiles)")print("observation:", C.N_SPATIAL_CHANNELS, "map layers +", C.N_SCALARS, "scalars")print("Q-net parameters:", f"{count_parameters(RCQNet(C.TrainConfig())):,}")

## 1. Look at a baseTown Hall in the middle, high-value defenses (X-Bow, Inferno, Scattershot,Monolith, Eagle) in the core, ordinary defenses in a ring, collectors pushedoutside, walls packing the centre. The red wash is the **ground** threat map —note the Air Defenses do not contribute to it, because they cannot touch aground unit.

In [ ]:
env = RCWalkEnv(defense_frac=1.0, seed=7)env.reset()fig, ax = plt.subplots(figsize=(9, 9))render_base(env, ax=ax, title="Generated TH15 base + ground threat")plt.show()from collections import Counterc = Counter(b.name for b in env.buildings if b.name != "Wall")print(dict(sorted(c.items(), key=lambda kv: -kv[1])))print("defenses:", sum(1 for b in env.buildings if b.is_defense),      "| Eagle awake:", env.eagle_active,      "| legal deploy cells:", int(env.legal_actions()[C.ACTION_TILE_OFFSET:].sum()))

## 2. Is the task actually winnable?Run this before any long training. It is the check the previous version neverhad: a scripted policy must be able to win, or nothing can.

In [ ]:
r = run_policy(policy_scripted_human, episodes=60, defense_frac=1.0, spells=8)print(f"scripted-human on full TH15:  Town Hall destroyed {100*r['th_kill_rate']:.0f}% "      f"of attacks | {r['stars']:.2f} stars | {100*r['destruction']:.0f}% destruction")assert r["th_kill_rate"] > 0.1, "task looks unwinnable -- do not start a long run"print("OK: the objective is reachable.")

## 3. TrainSafe to interrupt. `trainer.load()` restores weights, optimizer, epsilon,episode count and curriculum stage, so a resume is a real resume.For an overnight run, use the CLI instead — it survives the notebook dying:```bashpython -m coc.train --out runs/rc --hours 10 --resume --save-replay 20000```

In [ ]:
cfg = C.gpu_preset() if torch.cuda.is_available() else C.TrainConfig()cfg.max_hours = 0.5          # raise this for a real runcfg.eval_every = 250trainer = Trainer(cfg, out_dir=str(ROOT / "runs" / "rc"))trainer.load()               # no-op on a fresh runtrainer.train()

## 4. Learning curveIf these are flat, stop and diagnose — do not just run it longer.

In [ ]:
plot_training(str(ROOT / "runs" / "rc" / "metrics.csv"),              str(ROOT / "media" / "training_curve.png"))from IPython.display import ImageImage(str(ROOT / "media" / "training_curve.png"))

## 5. Evaluate against the baselinesThe trained agent has to beat `scripted-human` to be worth anything.

In [ ]:
results = compare(model_path=str(ROOT / "runs" / "rc" / "latest.pt"),                  episodes=100, defense_frac=1.0, spells=8)

## 6. Watch it attackBlue wash is active invisibility, the black dot is the Royal Champion, the line is her current target.

In [ ]:
dev = pick_device("auto")blob = torch.load(str(ROOT / "runs" / "rc" / "latest.pt"), map_location=dev, weights_only=False)net = RCQNet(C.TrainConfig(**{k: v for k, v in blob["cfg"].items()                              if k in C.TrainConfig.__dataclass_fields__})).to(dev)net.load_state_dict(blob["policy"]); net.eval()r = run_policy(make_model_policy(net, dev), episodes=25, defense_frac=1.0,               spells=8, record_best=True)path = animate_battle(r["frames"], str(ROOT / "media" / "battle_replay.gif"),                      title="Royal Champion walk — trained DQN")print("saved", path)from IPython.display import ImageImage(path)